# Notebook 02 — scRNA-seq QC and Cell Filtering

**Module:** 10 — Transcriptomics and Proteomics  
**Tier:** 2 — Working competence  
**Notebook:** 02 of 12  
**Time estimate:** 75 minutes

> Before you can interpret single-cell data, you must remove cells that are
> too damaged, too empty, or not cells at all.
> QC is not preprocessing — it is the difference between a real result and an artefact.

---
## Step 1 — Motivation

A 10x Genomics run captures ~10,000 barcodes, but perhaps only 2,000–5,000 correspond
to real cells. The rest are empty droplets (ambient RNA only), damaged cells, or
doublets. Including these destroys downstream clustering and differential expression.

---
## Step 2 — Intuition

Three QC metrics are computed per cell:
1. **total_counts** — total UMIs. Too low = empty/dead. Too high = doublet.
2. **n_genes_by_counts** — number of distinct genes detected. Too low = empty.
3. **pct_counts_mt** — fraction of UMIs from mitochondrial genes. Too high = dying cell.

The threshold question is: where is "too low" or "too high"?

**Fixed thresholds** (e.g. min 200 genes) are common but dataset-specific.
**MAD-based thresholds** are more principled: flag cells more than k MADs from the
median as outliers, where MAD = Median Absolute Deviation.

---
## Step 3 — Biological Background

**Why does mito fraction detect dying cells?**  
Cytoplasmic mRNA degrades rapidly when a cell dies. Mitochondria are membrane-enclosed
organelles — their mRNA survives longer. A dying cell shows depleted cytoplasmic mRNA
but intact mitochondrial mRNA, inflating pct_counts_mt.

**Why does high total_counts indicate doublets?**  
If two cells enter one droplet, both cells' mRNA is captured under one barcode. The
resulting "cell" has roughly double the library size and double the gene count of a
typical single cell. Computational doublet detection tools (Scrublet, DoubletFinder)
exploit this signature.

**Empty droplets:**  
Droplets without a cell still capture ambient RNA floating in the cell suspension.
They have very low UMI counts (typically <100). Cell Ranger's "filtered" output
uses an emptyDroplets-based method (Lun 2019) to exclude these.

---
## Step 4 — Mathematical Explanation

**MAD (Median Absolute Deviation):**
$$\text{MAD}(x) = \text{median}(|x_i - \text{median}(x)|)$$

MAD is a robust estimator of spread — unlike standard deviation, a few extreme outliers
don't inflate it. For QC, we typically work on log-scale:

$$\text{is\_outlier}(x_i) = x_i < \text{median}(x) - k \cdot \text{MAD}(x)$$

where $k = 3$ or $5$ depending on stringency (3 is typical for scRNA-seq).

For pct_counts_mt we use only the upper tail (high mito is bad, low mito is fine):
$$\text{is\_outlier\_mito}(x_i) = x_i > \text{median}(x) + k \cdot \text{MAD}(x)$$

**Why log-scale for total_counts and n_genes?**  
Library sizes are roughly log-normally distributed. Working in log-space makes the
distribution more symmetric, so MAD-based thresholds are better calibrated.

In [ ]:
# Step 6 — Python: QC metrics and MAD-based filtering
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# --- Rebuild the synthetic count matrix from NB01 ---
N_CELLS = 300
N_GENES = 1500
N_MITO_GENES = 13

cell_types_all = rng.choice(['T_cell', 'B_cell', 'Monocyte'], N_CELLS, p=[0.4, 0.35, 0.25])

def simulate_counts(n_cells, n_genes, n_mito, cell_types, rng,
                    n_damaged=15, n_doublets=10, n_empty=20):
    """Simulate count matrix with planted artefact cells."""
    X = np.zeros((n_cells, n_genes), dtype=np.int32)
    cell_class = ['real'] * n_cells
    markers = {
        'T_cell':   (list(range(0, 40)), 5),
        'B_cell':   (list(range(40, 80)), 6),
        'Monocyte': (list(range(80, 130)), 4),
    }
    for i, ct in enumerate(cell_types):
        mg, boost = markers[ct]
        base = rng.exponential(0.5, n_genes)
        base[mg] *= boost
        lib = int(np.clip(rng.normal(2500, 400), 500, 8000))
        X[i, :] = rng.multinomial(lib, base / base.sum())

    # Plant damaged cells (high mito)
    dam_idx = rng.choice(n_cells, n_damaged, replace=False)
    for i in dam_idx:
        lib = int(rng.normal(1000, 200))
        lib = max(200, lib)
        base = np.ones(n_genes) * 0.01
        base[-n_mito:] = 3.0  # mostly mitochondrial
        X[i, :] = rng.multinomial(lib, base / base.sum())
        cell_class[i] = 'damaged'

    # Plant doublets (high total counts)
    dbl_idx = rng.choice([j for j in range(n_cells) if cell_class[j] == 'real'],
                          n_doublets, replace=False)
    for i in dbl_idx:
        lib = int(rng.normal(6000, 500))  # ~2x normal
        base = rng.exponential(0.5, n_genes)
        X[i, :] = rng.multinomial(lib, base / base.sum())
        cell_class[i] = 'doublet'

    # Plant empty droplets (very low counts)
    emp_idx = rng.choice([j for j in range(n_cells) if cell_class[j] == 'real'],
                          n_empty, replace=False)
    for i in emp_idx:
        lib = int(rng.normal(80, 20))
        lib = max(10, lib)
        base = rng.exponential(0.5, n_genes)
        X[i, :] = rng.multinomial(lib, base / base.sum())
        cell_class[i] = 'empty'

    return X, np.array(cell_class)

X, cell_class = simulate_counts(N_CELLS, N_GENES, N_MITO_GENES, cell_types_all, rng)

# --- Compute QC metrics ---
def compute_qc_metrics(X, n_mito_genes):
    """Compute per-cell QC metrics."""
    total_counts = X.sum(axis=1)
    n_genes_by_counts = (X > 0).sum(axis=1)
    mito_counts = X[:, -n_mito_genes:].sum(axis=1)
    pct_counts_mt = np.where(total_counts > 0,
                              mito_counts / total_counts * 100, 0)
    return {
        'total_counts': total_counts,
        'n_genes_by_counts': n_genes_by_counts,
        'pct_counts_mt': pct_counts_mt,
    }

qc = compute_qc_metrics(X, N_MITO_GENES)

# --- MAD-based outlier detection ---
def mad(x):
    return np.median(np.abs(x - np.median(x)))

def is_outlier_low(x, nmads=3):
    """Flag values more than nmads MADs below the median (log-scale)."""
    lx = np.log1p(x)
    return lx < np.median(lx) - nmads * mad(lx)

def is_outlier_high(x, nmads=3):
    """Flag values more than nmads MADs above the median."""
    return x > np.median(x) + nmads * mad(x)

outlier_total_low   = is_outlier_low(qc['total_counts'])
outlier_total_high  = is_outlier_high(qc['total_counts'])
outlier_genes_low   = is_outlier_low(qc['n_genes_by_counts'])
outlier_mito_high   = is_outlier_high(qc['pct_counts_mt'])

remove = outlier_total_low | outlier_total_high | outlier_genes_low | outlier_mito_high
keep_mask = ~remove

print('QC summary:')
print(f'  Total cells:              {N_CELLS}')
print(f'  Low total counts:         {outlier_total_low.sum()}')
print(f'  High total counts:        {outlier_total_high.sum()}')
print(f'  Low n_genes:              {outlier_genes_low.sum()}')
print(f'  High mito %:              {outlier_mito_high.sum()}')
print(f'  Cells removed:            {remove.sum()}')
print(f'  Cells retained:           {keep_mask.sum()}')
print()
print('Ground truth in removed cells:')
for cls in ['empty', 'damaged', 'doublet', 'real']:
    n_removed = (remove & (cell_class == cls)).sum()
    n_total = (cell_class == cls).sum()
    print(f'  {cls:12s}: {n_removed}/{n_total} removed')

In [ ]:
# Step 7 — Visualization: QC metric distributions and filtering decisions
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

colors_cls = {'real': 'steelblue', 'damaged': 'tomato',
               'doublet': 'orange', 'empty': 'grey'}

# Panel A: Library size distribution
ax = axes[0, 0]
for cls, col in colors_cls.items():
    mask = cell_class == cls
    ax.scatter(np.arange(N_CELLS)[mask], qc['total_counts'][mask],
                c=col, s=10, alpha=0.6, label=cls)
ax.set_xlabel('Cell index')
ax.set_ylabel('Total UMI counts')
ax.set_title('A. Library size by true cell class')
ax.legend(fontsize=7, markerscale=2)

# Panel B: n_genes vs total_counts scatter
ax = axes[0, 1]
for cls, col in colors_cls.items():
    mask = cell_class == cls
    ax.scatter(qc['total_counts'][mask], qc['n_genes_by_counts'][mask],
                c=col, s=10, alpha=0.6, label=cls)
ax.set_xlabel('Total UMI counts')
ax.set_ylabel('Genes detected')
ax.set_title('B. Genes vs. library size')
ax.legend(fontsize=7, markerscale=2)

# Panel C: Mito % vs total counts
ax = axes[0, 2]
for cls, col in colors_cls.items():
    mask = cell_class == cls
    ax.scatter(qc['total_counts'][mask], qc['pct_counts_mt'][mask],
                c=col, s=10, alpha=0.6, label=cls)
ax.set_xlabel('Total UMI counts')
ax.set_ylabel('% Mitochondrial UMIs')
ax.set_title('C. Mito fraction vs. library size')
ax.legend(fontsize=7, markerscale=2)

# Panel D: MAD threshold on log(total_counts)
ax = axes[1, 0]
log_tc = np.log1p(qc['total_counts'])
med_tc = np.median(log_tc)
mad_tc = mad(log_tc)
ax.hist(log_tc, bins=40, color='lightblue', edgecolor='none')
ax.axvline(med_tc - 3 * mad_tc, color='red', ls='--', label='median ± 3 MAD')
ax.axvline(med_tc + 3 * mad_tc, color='red', ls='--')
ax.axvline(med_tc, color='black', ls='-', alpha=0.4, label='median')
ax.set_xlabel('log(total_counts + 1)')
ax.set_title('D. MAD threshold on library size\n(3 MADs from median)')
ax.legend(fontsize=8)

# Panel E: Before/after filtering
ax = axes[1, 1]
before_class_counts = {cls: (cell_class == cls).sum() for cls in colors_cls}
after_class_counts = {cls: (keep_mask & (cell_class == cls)).sum() for cls in colors_cls}
x = np.arange(len(colors_cls))
bars_b = ax.bar(x - 0.2, [before_class_counts[c] for c in colors_cls],
                  width=0.35, label='Before QC', color=[colors_cls[c] for c in colors_cls], alpha=0.9)
bars_a = ax.bar(x + 0.2, [after_class_counts[c] for c in colors_cls],
                  width=0.35, label='After QC', color=[colors_cls[c] for c in colors_cls], alpha=0.4, hatch='//')
ax.set_xticks(x)
ax.set_xticklabels(list(colors_cls.keys()))
ax.set_ylabel('Number of cells')
ax.set_title('E. Cells before vs. after QC filtering')
ax.legend(fontsize=8)

# Panel F: Violin of QC metrics post-filter
ax = axes[1, 2]
data_to_plot = [
    np.log1p(qc['total_counts'][keep_mask]),
    np.log1p(qc['n_genes_by_counts'][keep_mask]),
]
vp = ax.violinplot(data_to_plot, positions=[1, 2], showmedians=True)
ax.set_xticks([1, 2])
ax.set_xticklabels(['log(total_counts)', 'log(n_genes)'])
ax.set_title('F. QC distributions after filtering')
ax.set_ylabel('log-scale value')

plt.suptitle('Module 10 NB02: scRNA-seq QC and Cell Filtering', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('scrna_qc.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Write `mad_threshold(x, nmads, log_scale=True)` → returns `(lower, upper)` thresholds.
2. What happens if you use `nmads=5` instead of `3`? Run it and report the change in
   cells removed.
3. Why do we apply MAD on the log scale for `total_counts` but not for `pct_counts_mt`?
4. A cell has 4500 UMI counts and 25% mitochondrial fraction. Should it be removed?
   Defend your answer with reference to the thresholds computed above.

---
## Step 10 — Quiz

1. Name the three standard QC metrics for scRNA-seq cell filtering.
2. What is the MAD? Why is it preferred over standard deviation for outlier detection?
3. Why does a doublet have a high library size?
4. Scan-py's `sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'])` — what does
   `qc_vars=['mt']` do?
5. After filtering, what fraction of the ground-truth 'real' cells were incorrectly
   removed in our simulation?

---
## Step 12 — Reflection

> *[In one paragraph: explain why QC filtering in scRNA-seq cannot be reduced
> to a single universal threshold, and what the consequences are of being too
> permissive vs. too strict.]*

---
*Next: `03_normalization_and_log_transformation.ipynb`*